<a href="https://colab.research.google.com/github/BotCalvin/BUS-118S/blob/main/Group_9_Exercise_Agentic_AI_in_Supply_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm


# -----------------------------
# GLOBAL SETTINGS / PARAMETERS
# -----------------------------
AUTO_GENERATE_DATA = True
RANDOM_SEED = 42

FORECAST_ALPHA = 0.30
REVIEW_PERIOD_DAYS = 1
DEFAULT_ORDER_COST = 0.0
DEFAULT_MAX_COVER_DAYS = 14   # guardrail to prevent unrealistic over-ordering
PRINT_DAILY_LOG_PREVIEW = True
DAILY_LOG_PREVIEW_ROWS = 20

np.random.seed(RANDOM_SEED)


# =========================================================
# 1. DATA GENERATION
# =========================================================
def generate_input_csvs(
    start_date="2024-01-01",
    periods=90,
    skus=("SKU_A", "SKU_B", "SKU_C")
):
    """
    Auto-generate sales.csv, inventory.csv, and params.csv
    using the schemas required by the assignment.
    """
    dates = pd.date_range(start=start_date, periods=periods, freq="D")

    sales_rows = []
    for sku in skus:
        if sku == "SKU_A":
            base_demand = 18
            demand_std = 4
        elif sku == "SKU_B":
            base_demand = 12
            demand_std = 3
        else:
            base_demand = 8
            demand_std = 2

        for date in dates:
            qty_sold = max(0, int(np.random.normal(loc=base_demand, scale=demand_std)))
            sales_rows.append([date, sku, qty_sold])

    sales_df = pd.DataFrame(sales_rows, columns=["date", "sku", "qty_sold"])
    sales_df.to_csv("sales.csv", index=False)

    inventory_df = pd.DataFrame({
        "sku": list(skus),
        "opening_stock": [120, 90, 70]
    })
    inventory_df.to_csv("inventory.csv", index=False)

    # Added optional columns:
    # - order_cost: per-PO cost (can be 0)
    # - inventory_cap: guardrail against unrealistic over-ordering
    params_df = pd.DataFrame({
        "sku": list(skus),
        "unit_cost": [10, 8, 6],
        "holding_cost_per_day": [0.10, 0.08, 0.06],
        "stockout_cost": [5, 4, 3],
        "lead_time_days": [3, 4, 2],
        "min_order_qty": [30, 25, 20],
        "service_level": [0.95, 0.90, 0.92],
        "order_cost": [0, 0, 0],
        "inventory_cap": [300, 220, 180]
    })
    params_df.to_csv("params.csv", index=False)

    print("Generated input files:")
    print("- sales.csv")
    print("- inventory.csv")
    print("- params.csv")


# =========================================================
# 2. DATA LOADING / VALIDATION
# =========================================================
def load_and_prepare_data():
    """
    Load all three CSV files and validate required columns.
    """
    sales = pd.read_csv("sales.csv", parse_dates=["date"])
    inventory = pd.read_csv("inventory.csv")
    params = pd.read_csv("params.csv")

    required_sales = {"date", "sku", "qty_sold"}
    required_inventory = {"sku", "opening_stock"}
    required_params = {
        "sku", "unit_cost", "holding_cost_per_day", "stockout_cost",
        "lead_time_days", "min_order_qty", "service_level"
    }

    assert required_sales.issubset(sales.columns), "sales.csv is missing required columns."
    assert required_inventory.issubset(inventory.columns), "inventory.csv is missing required columns."
    assert required_params.issubset(params.columns), "params.csv is missing required columns."

    # Optional columns with defaults
    if "order_cost" not in params.columns:
        params["order_cost"] = DEFAULT_ORDER_COST
    if "inventory_cap" not in params.columns:
        params["inventory_cap"] = np.nan

    sales = sales.sort_values(["sku", "date"])

    # Aggregate demand per SKU per day (rubric requirement)
    sales = (
        sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
        .sum()
        .sort_values(["sku", "date"])
    )

    inventory_full = inventory.merge(params, on="sku", how="inner")

    return sales, inventory_full


# =========================================================
# 3. FORECASTING / SAFETY STOCK HELPERS
# =========================================================
def ewma_forecast(series, alpha=0.30):
    """
    Generate one-step-ahead EWMA forecast series.
    """
    if len(series) == 0:
        return pd.Series(dtype=float)

    forecast = [series.iloc[0]]
    for t in range(1, len(series)):
        next_forecast = alpha * series.iloc[t - 1] + (1 - alpha) * forecast[-1]
        forecast.append(next_forecast)

    return pd.Series(forecast, index=series.index)


def get_z_value(service_level):
    """
    Convert service level to z-score.
    """
    safe_service = min(max(service_level, 0.50), 0.999)
    return norm.ppf(safe_service)


def compute_safety_stock(actual_series, forecast_series, lead_time_days, service_level):
    """
    Safety stock = z * forecast error std * sqrt(lead time)
    """
    errors = actual_series - forecast_series
    error_std = errors.std(ddof=1)

    if pd.isna(error_std) or error_std == 0:
        error_std = 1e-6

    z_value = get_z_value(service_level)
    safety_stock = max(0.0, z_value * error_std * np.sqrt(max(1, lead_time_days)))
    return safety_stock, error_std


# =========================================================
# 4. POLICY RULES
# =========================================================
def agent_policy(
    stock_on_hand,
    on_order_qty,
    forecast_value,
    safety_stock,
    lead_time_days,
    review_period_days,
    min_order_qty,
    inventory_cap=None
):
    """
    Forecast-based replenishment agent.
    Projects demand over lead time + review period.
    Orders up to target level if inventory position is below reorder point.
    """
    coverage_days = lead_time_days + review_period_days
    expected_demand = forecast_value * coverage_days
    reorder_point = expected_demand + safety_stock
    target_level = reorder_point

    inventory_position = stock_on_hand + on_order_qty

    if inventory_position < reorder_point:
        raw_order_qty = int(np.ceil(target_level - inventory_position))
        order_qty = max(raw_order_qty, int(min_order_qty))

        # Guardrail: do not exceed inventory cap if one exists
        if pd.notna(inventory_cap):
            max_allowed_order = max(0, int(inventory_cap - inventory_position))
            order_qty = min(order_qty, max_allowed_order)

        if order_qty > 0:
            reason = (
                f"Placed order because inventory position ({inventory_position:.1f}) "
                f"is below reorder point ({reorder_point:.1f})."
            )
            return True, order_qty, reorder_point, target_level, reason

    reason = (
        f"No order because inventory position ({inventory_position:.1f}) "
        f"is sufficient versus reorder point ({reorder_point:.1f})."
    )
    return False, 0, reorder_point, target_level, reason


def baseline_policy(
    stock_on_hand,
    on_order_qty,
    avg_daily_demand,
    lead_time_days,
    min_order_qty,
    inventory_cap=None
):
    """
    Simple baseline policy:
    If inventory position falls below avg_daily_demand * lead_time,
    place min_order_qty only.
    """
    baseline_reorder_point = avg_daily_demand * lead_time_days
    inventory_position = stock_on_hand + on_order_qty

    if inventory_position < baseline_reorder_point:
        order_qty = int(min_order_qty)

        if pd.notna(inventory_cap):
            max_allowed_order = max(0, int(inventory_cap - inventory_position))
            order_qty = min(order_qty, max_allowed_order)

        if order_qty > 0:
            reason = (
                f"Baseline ordered min qty because inventory position ({inventory_position:.1f}) "
                f"is below baseline reorder point ({baseline_reorder_point:.1f})."
            )
            return True, order_qty, baseline_reorder_point, reason

    reason = (
        f"Baseline waited because inventory position ({inventory_position:.1f}) "
        f"is sufficient versus baseline reorder point ({baseline_reorder_point:.1f})."
    )
    return False, 0, baseline_reorder_point, reason


# =========================================================
# 5. CORE SIMULATION
# =========================================================
def simulate_policy(
    sales,
    inventory_full,
    policy_name="agent",
    alpha=0.30,
    review_period_days=1
):
    """
    Simulate daily inventory operations for each SKU under a given policy.
    Returns summary metrics and detailed daily logs.
    """
    summary_rows = []
    daily_logs = []

    for _, inv in inventory_full.iterrows():
        sku = inv["sku"]

        sku_sales = sales[sales["sku"] == sku].copy()
        sku_sales = sku_sales.set_index("date").asfreq("D")
        sku_sales["sku"] = sku
        sku_sales["qty_sold"] = sku_sales["qty_sold"].fillna(0)

        # Parameters
        opening_stock = float(inv["opening_stock"])
        holding_cost_per_day = float(inv["holding_cost_per_day"])
        stockout_cost = float(inv["stockout_cost"])
        lead_time_days = int(inv["lead_time_days"])
        min_order_qty = int(inv["min_order_qty"])
        service_level = float(inv["service_level"])
        order_cost = float(inv.get("order_cost", DEFAULT_ORDER_COST))
        inventory_cap = inv.get("inventory_cap", np.nan)

        # Forecast
        sku_sales["forecast"] = ewma_forecast(sku_sales["qty_sold"], alpha=alpha)

        # Safety stock based on full-series forecast error
        safety_stock, forecast_error_std = compute_safety_stock(
            sku_sales["qty_sold"],
            sku_sales["forecast"],
            lead_time_days,
            service_level
        )

        # Baseline helper
        avg_daily_demand = sku_sales["qty_sold"].mean()

        # Simulation state
        stock = opening_stock
        pipeline_orders = []

        total_demand = 0.0
        total_fulfilled = 0.0
        total_stockouts = 0.0
        total_holding_cost = 0.0
        total_stockout_cost = 0.0
        total_order_cost = 0.0
        orders_placed = 0

        for day in sku_sales.index:
            demand = float(sku_sales.loc[day, "qty_sold"])
            forecast_value = float(sku_sales.loc[day, "forecast"])

            total_demand += demand

            # Receive arriving POs
            arrivals_today = [po for po in pipeline_orders if po["arrival_date"] == day]
            qty_received = sum(po["qty"] for po in arrivals_today)
            stock += qty_received

            pipeline_orders = [po for po in pipeline_orders if po["arrival_date"] > day]

            # Fulfill demand
            fulfilled = min(stock, demand)
            unmet = demand - fulfilled
            stock -= fulfilled

            total_fulfilled += fulfilled
            total_stockouts += unmet
            total_stockout_cost += unmet * stockout_cost

            # Holding cost assessed on ending stock
            total_holding_cost += stock * holding_cost_per_day

            # Current on-order qty after arrivals
            on_order_qty = sum(po["qty"] for po in pipeline_orders)

            # Choose policy
            if policy_name == "agent":
                order_placed, order_qty, reorder_point, target_level, decision_reason = agent_policy(
                    stock_on_hand=stock,
                    on_order_qty=on_order_qty,
                    forecast_value=forecast_value,
                    safety_stock=safety_stock,
                    lead_time_days=lead_time_days,
                    review_period_days=review_period_days,
                    min_order_qty=min_order_qty,
                    inventory_cap=inventory_cap
                )
            elif policy_name == "baseline":
                order_placed, order_qty, reorder_point, decision_reason = baseline_policy(
                    stock_on_hand=stock,
                    on_order_qty=on_order_qty,
                    avg_daily_demand=avg_daily_demand,
                    lead_time_days=lead_time_days,
                    min_order_qty=min_order_qty,
                    inventory_cap=inventory_cap
                )
                target_level = reorder_point
            else:
                raise ValueError("policy_name must be either 'agent' or 'baseline'.")

            # Create PO if needed
            if order_placed and order_qty > 0:
                arrival_date = day + pd.Timedelta(days=lead_time_days)
                pipeline_orders.append({
                    "qty": order_qty,
                    "arrival_date": arrival_date
                })
                orders_placed += 1
                total_order_cost += order_cost

            inventory_position = stock + sum(po["qty"] for po in pipeline_orders)

            daily_logs.append({
                "policy": policy_name,
                "date": day,
                "sku": sku,
                "demand": demand,
                "forecast": round(forecast_value, 2),
                "safety_stock": round(safety_stock, 2),
                "forecast_error_std": round(forecast_error_std, 2),
                "qty_received_today": qty_received,
                "ending_stock": round(stock, 2),
                "on_order_qty": round(sum(po["qty"] for po in pipeline_orders), 2),
                "inventory_position": round(inventory_position, 2),
                "reorder_point": round(reorder_point, 2),
                "target_level": round(target_level, 2),
                "order_placed": order_placed,
                "order_qty": order_qty,
                "unmet_demand": round(unmet, 2),
                "decision_reason": decision_reason
            })

        fill_rate = total_fulfilled / total_demand if total_demand > 0 else 1.0
        total_cost = total_holding_cost + total_stockout_cost + total_order_cost

        summary_rows.append({
            "policy": policy_name,
            "sku": sku,
            "total_demand": round(total_demand, 2),
            "fulfilled_demand": round(total_fulfilled, 2),
            "stockouts": round(total_stockouts, 2),
            "fill_rate": round(fill_rate, 4),
            "orders_placed": orders_placed,
            "holding_cost": round(total_holding_cost, 2),
            "stockout_cost": round(total_stockout_cost, 2),
            "order_cost": round(total_order_cost, 2),
            "total_cost": round(total_cost, 2)
        })

    summary_df = pd.DataFrame(summary_rows)
    daily_log_df = pd.DataFrame(daily_logs)

    return summary_df, daily_log_df


# =========================================================
# 6. COMPARISON / REPORTING
# =========================================================
def compare_policies(agent_summary, baseline_summary):
    """
    Compare agent vs baseline at the SKU level and in total.
    """
    compare_df = agent_summary.merge(
        baseline_summary,
        on="sku",
        suffixes=("_agent", "_baseline")
    )

    compare_df["fill_rate_improvement"] = (
        compare_df["fill_rate_agent"] - compare_df["fill_rate_baseline"]
    )
    compare_df["cost_savings"] = (
        compare_df["total_cost_baseline"] - compare_df["total_cost_agent"]
    )
    compare_df["stockout_reduction"] = (
        compare_df["stockouts_baseline"] - compare_df["stockouts_agent"]
    )

    total_row = pd.DataFrame([{
        "sku": "TOTAL",
        "total_demand_agent": compare_df["total_demand_agent"].sum(),
        "fulfilled_demand_agent": compare_df["fulfilled_demand_agent"].sum(),
        "stockouts_agent": compare_df["stockouts_agent"].sum(),
        "fill_rate_agent": round(
            compare_df["fulfilled_demand_agent"].sum() / max(compare_df["total_demand_agent"].sum(), 1),
            4
        ),
        "orders_placed_agent": compare_df["orders_placed_agent"].sum(),
        "holding_cost_agent": compare_df["holding_cost_agent"].sum(),
        "stockout_cost_agent": compare_df["stockout_cost_agent"].sum(),
        "order_cost_agent": compare_df["order_cost_agent"].sum(),
        "total_cost_agent": compare_df["total_cost_agent"].sum(),

        "total_demand_baseline": compare_df["total_demand_baseline"].sum(),
        "fulfilled_demand_baseline": compare_df["fulfilled_demand_baseline"].sum(),
        "stockouts_baseline": compare_df["stockouts_baseline"].sum(),
        "fill_rate_baseline": round(
            compare_df["fulfilled_demand_baseline"].sum() / max(compare_df["total_demand_baseline"].sum(), 1),
            4
        ),
        "orders_placed_baseline": compare_df["orders_placed_baseline"].sum(),
        "holding_cost_baseline": compare_df["holding_cost_baseline"].sum(),
        "stockout_cost_baseline": compare_df["stockout_cost_baseline"].sum(),
        "order_cost_baseline": compare_df["order_cost_baseline"].sum(),
        "total_cost_baseline": compare_df["total_cost_baseline"].sum(),

        "fill_rate_improvement": round(
            (compare_df["fulfilled_demand_agent"].sum() / max(compare_df["total_demand_agent"].sum(), 1)) -
            (compare_df["fulfilled_demand_baseline"].sum() / max(compare_df["total_demand_baseline"].sum(), 1)),
            4
        ),
        "cost_savings": round(
            compare_df["total_cost_baseline"].sum() - compare_df["total_cost_agent"].sum(),
            2
        ),
        "stockout_reduction": round(
            compare_df["stockouts_baseline"].sum() - compare_df["stockouts_agent"].sum(),
            2
        )
    }])

    compare_df = pd.concat([compare_df, total_row], ignore_index=True)
    return compare_df


def print_tradeoff_reflection(compare_df):
    """
    Brief trade-off explanation for the rubric.
    """
    total_row = compare_df[compare_df["sku"] == "TOTAL"].iloc[0]

    print("\n=== TRADE-OFF REFLECTION ===")
    print(
        "Higher inventory generally improves service levels and reduces stockouts, "
        "but it increases holding cost. Lower inventory reduces carrying cost, "
        "but it raises the risk of missed demand and customer service issues."
    )
    print(
        f"In this run, the agent changed fill rate by {total_row['fill_rate_improvement']:.4f}, "
        f"reduced stockouts by {total_row['stockout_reduction']:.2f}, "
        f"and changed total cost by ${total_row['cost_savings']:.2f} versus the baseline."
    )


# =========================================================
# 7. MAIN EXECUTION
# =========================================================
def main():
    if AUTO_GENERATE_DATA:
        generate_input_csvs()

    sales, inventory_full = load_and_prepare_data()

    # Agent simulation
    agent_summary, agent_log = simulate_policy(
        sales=sales,
        inventory_full=inventory_full,
        policy_name="agent",
        alpha=FORECAST_ALPHA,
        review_period_days=REVIEW_PERIOD_DAYS
    )

    # Baseline simulation
    baseline_summary, baseline_log = simulate_policy(
        sales=sales,
        inventory_full=inventory_full,
        policy_name="baseline",
        alpha=FORECAST_ALPHA,
        review_period_days=REVIEW_PERIOD_DAYS
    )

    # Compare results
    comparison_df = compare_policies(agent_summary, baseline_summary)

    # Save outputs
    agent_summary.to_csv("agent_summary.csv", index=False)
    baseline_summary.to_csv("baseline_summary.csv", index=False)
    agent_log.to_csv("agent_daily_log.csv", index=False)
    baseline_log.to_csv("baseline_daily_log.csv", index=False)
    comparison_df.to_csv("policy_comparison.csv", index=False)

    # Print main outputs
    print("\n=== AGENT SUMMARY ===")
    print(agent_summary.to_string(index=False))

    print("\n=== BASELINE SUMMARY ===")
    print(baseline_summary.to_string(index=False))

    print("\n=== POLICY COMPARISON ===")
    print(comparison_df.to_string(index=False))

    if PRINT_DAILY_LOG_PREVIEW:
        print("\n=== AGENT DAILY LOG PREVIEW ===")
        print(agent_log.head(DAILY_LOG_PREVIEW_ROWS).to_string(index=False))

    print_tradeoff_reflection(comparison_df)

    print("\nFiles created:")
    print("- sales.csv")
    print("- inventory.csv")
    print("- params.csv")
    print("- agent_summary.csv")
    print("- baseline_summary.csv")
    print("- agent_daily_log.csv")
    print("- baseline_daily_log.csv")
    print("- policy_comparison.csv")


# Run the notebook
main()

Generated input files:
- sales.csv
- inventory.csv
- params.csv

=== AGENT SUMMARY ===
policy   sku  total_demand  fulfilled_demand  stockouts  fill_rate  orders_placed  holding_cost  stockout_cost  order_cost  total_cost
 agent SKU_A        1543.0            1543.0        0.0        1.0             51        405.20            0.0         0.0      405.20
 agent SKU_B        1049.0            1049.0        0.0        1.0             42        232.08            0.0         0.0      232.08
 agent SKU_C         678.0             678.0        0.0        1.0             32        128.46            0.0         0.0      128.46

=== BASELINE SUMMARY ===
  policy   sku  total_demand  fulfilled_demand  stockouts  fill_rate  orders_placed  holding_cost  stockout_cost  order_cost  total_cost
baseline SKU_A        1543.0            1520.0       23.0     0.9851             49        173.20          115.0         0.0      288.20
baseline SKU_B        1049.0            1035.0       14.0     0.9867     